In [0]:
from pyspark.sql import functions as F

# Read the validated Silver Delta table
silver_df = spark.table("workspace.default.silver_wind_power")

# Verify the source data
print("Number of rows:", silver_df.count())
print("Number of columns:", len(silver_df.columns))

display(silver_df.limit(10))

In [0]:
# Calculate daily production metrics for each turbine
gold_daily_df = (
    silver_df
    .groupBy("date", "turbine_name")
    .agg(
        F.sum("energy_produced").alias("total_energy_produced"),
        F.avg("wind_speed").alias("avg_wind_speed"),
        F.count("*").alias("total_measurements"),
        F.sum(
            F.when(F.col("status") == "Online", 1).otherwise(0)
        ).alias("online_measurements"),

        F.sum(
            F.when(F.col("status") == "Offline", 1).otherwise(0)
        ).alias("offline_measurements")
    )
    .orderBy("date", "turbine_name")
)

display(gold_daily_df.limit(10))

In [0]:
# Validate the aggregated dataset
print("Number of rows:", gold_daily_df.count())
print("Number of columns:", len(gold_daily_df.columns))

display(gold_daily_df)

In [0]:
print("Number of rows:", gold_daily_df.count())
print("Number of columns:", len(gold_daily_df.columns))

display(gold_daily_df)

In [0]:
# Save daily production metrics as a Delta table

gold_daily_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_daily_production")

In [0]:
# Read the saved Gold table

gold_saved_df = spark.table("workspace.default.gold_daily_production")

print("Number of rows:", gold_saved_df.count())
print("Number of columns:", len(gold_saved_df.columns))

display(gold_saved_df.limit(10))

In [0]:
# Save daily production metrics as a Delta table

gold_daily_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_daily_production")

In [0]:
# Read the saved Gold table

gold_saved_df = spark.table("workspace.default.gold_daily_production")

print("Number of rows:", gold_saved_df.count())
print("Number of columns:", len(gold_saved_df.columns))

display(gold_saved_df.limit(10))

In [0]:
gold_daily_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_daily_production")

In [0]:
gold_saved_df = spark.table("workspace.default.gold_daily_production")

In [0]:
# Check Gold table dimensions

print("Number of rows:", gold_saved_df.count())
print("Number of columns:", len(gold_saved_df.columns))

In [0]:
# Check uniqueness of date and turbine_name

duplicate_groups_df = (
    gold_saved_df
    .groupBy("date", "turbine_name")
    .count()
    .filter(F.col("count") > 1)
)

print("Number of duplicate groups:", duplicate_groups_df.count())

display(duplicate_groups_df)

In [0]:
# Check measurement count consistency

inconsistent_measurements_df = gold_saved_df.filter(
    F.col("online_measurements")
    + F.col("offline_measurements")
    > F.col("total_measurements")
)

print(
    "Number of inconsistent groups:",
    inconsistent_measurements_df.count()
)

display(inconsistent_measurements_df)

In [0]:
# Calculate the percentage of measurements recorded as Online

gold_daily_df = gold_daily_df.withColumn(
    "online_measurement_pct",
    F.round(
        F.col("online_measurements")
        / F.col("total_measurements") * 100,
        2
    )
)

display(gold_daily_df.limit(10))

In [0]:
print("Number of rows:", gold_daily_df.count())
print("Number of columns:", len(gold_daily_df.columns))

display(
    gold_daily_df.select(
        "date",
        "turbine_name",
        "total_measurements",
        "online_measurements",
        "online_measurement_pct"
    ).limit(10)
)

In [0]:
gold_daily_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_daily_production")

In [0]:
%sql
ALTER TABLE workspace.default.gold_daily_production
ADD COLUMNS (online_measurement_pct DOUBLE)

In [0]:
gold_daily_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_daily_production")

In [0]:
gold_saved_df = spark.table(
    "workspace.default.gold_daily_production"
)

print("Number of rows:", gold_saved_df.count())
print("Number of columns:", len(gold_saved_df.columns))

display(
    gold_saved_df.select(
        "date",
        "turbine_name",
        "total_measurements",
        "online_measurements",
        "online_measurement_pct"
    ).limit(10)
)

In [0]:
gold_turbine_df = (
    silver_df
    .groupBy("turbine_name")
    .agg(
        F.sum("energy_produced").alias("total_energy_produced"),
        F.avg("wind_speed").alias("avg_wind_speed"),
        F.count("*").alias("total_measurements"),
        F.sum(
            F.when(F.col("status") == "Online", 1).otherwise(0)
        ).alias("online_measurements"),
        F.sum(
            F.when(F.col("status") == "Offline", 1).otherwise(0)
        ).alias("offline_measurements")
    )
    .withColumn(
        "online_measurement_pct",
        F.round(
            F.col("online_measurements")
            / F.col("total_measurements") * 100,
            2
        )
    )
    .orderBy(F.col("total_energy_produced").desc())
)

display(gold_turbine_df)

In [0]:
gold_turbine_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_turbine_performance")

In [0]:
gold_turbine_saved_df = spark.table(
    "workspace.default.gold_turbine_performance"
)

print("Nombre de lignes :", gold_turbine_saved_df.count())
print("Nombre de colonnes :", len(gold_turbine_saved_df.columns))

display(gold_turbine_saved_df)

In [0]:
duplicate_turbines_df = (
    gold_turbine_saved_df
    .groupBy("turbine_name")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Nombre de groupes de doublons :",
    duplicate_turbines_df.count()
)

display(duplicate_turbines_df)

In [0]:
duplicate_turbines_df = (
    gold_turbine_saved_df
    .groupBy("turbine_name")
    .count()
    .filter(F.col("count") > 1)
)

In [0]:
print(
    "Nombre de groupes de doublons :",
    duplicate_turbines_df.count()
)

In [0]:
display(duplicate_turbines_df)

In [0]:
# List tables in the default schema
display(spark.sql("SHOW TABLES IN workspace.default"))

In [0]:
# Inspect the schema of each table

tables = [
    "workspace.default.bronze_wind_power",
    "workspace.default.silver_wind_power",
    "workspace.default.gold_daily_production",
    "workspace.default.gold_turbine_performance"
]

for table_name in tables:
    print(f"\n{'=' * 60}")
    print(f"Table: {table_name}")
    print("=" * 60)

    spark.table(table_name).printSchema()

In [0]:
from pyspark.sql import functions as F
quality_history_df = spark.table(
    "workspace.default.data_quality_results"
)

display(
    quality_history_df
    .orderBy(F.col("execution_timestamp").desc_nulls_last())
)

In [0]:
from pyspark.sql import functions as F

quality_history_df = spark.table(
    "workspace.default.data_quality_results"
)

# Identify the latest quality check execution
latest_execution = (
    quality_history_df
    .filter(F.col("execution_timestamp").isNotNull())
    .agg(F.max("execution_timestamp").alias("latest_execution"))
    .first()["latest_execution"]
)

print("Latest quality check execution:", latest_execution)

# Keep only the results from that execution
latest_quality_results_df = quality_history_df.filter(
    F.col("execution_timestamp") == F.lit(latest_execution)
)

display(latest_quality_results_df)

In [0]:
# Count the quality checks and failures
number_of_rules = latest_quality_results_df.count()

failed_rules = latest_quality_results_df.filter(
    F.col("result") != "PASS"
).count()

print("Number of quality rules:", number_of_rules)
print("Failed quality rules:", failed_rules)

# Block Gold processing if quality checks fail
if number_of_rules != 8:
    raise ValueError(
        f"Expected 8 quality rules, but found {number_of_rules}."
    )

if failed_rules > 0:
    raise ValueError(
        f"Gold processing blocked: {failed_rules} quality rule(s) failed."
    )

print("Data quality validation successful. Gold processing can continue.")